# Skin Disease Image Classifier — Google Colab

`Input image → Preprocessing (resize / normalize / enhance) → CNN classifier → Predicted class`

**Classes:** `acne`, `eczema`, `rash`, `wart`, `birthmark`
**Data:** [DermNet](https://www.kaggle.com/datasets/shubhamgoel27/dermnet) — no Kaggle login needed.
**Model:** EfficientNetV2B0, ImageNet weights, two-phase transfer learning.

---

### What changed from the first version, and why

| Change | Reason |
|---|---|
| EfficientNetV2B0 replaces MobileNetV2 | ~4–6 points more accuracy at similar T4 cost |
| Fine-tune LR raised 1e-5 → 1e-4 with cosine decay | 1e-5 barely moved the backbone in 15 epochs |
| Unfreeze from block ~1/3 instead of layer 100 | more capacity to adapt away from ImageNet |
| Ambiguous source folders dropped (see Step 1) | the single biggest accuracy lever — details below |
| Disk-backed `tf.data` cache, batch 64, mixed precision | GPU was starved waiting on JPEG decode |
| Test-time augmentation at eval | ~1–2 points, free |

**On the accuracy target.** 70–80% on five classes is a reasonable goal, but it is not
something a notebook can promise — it depends on how cleanly the classes separate.
The honest lever is class definition, not architecture: the first version's `rash`
bucket included contact dermatitis / poison ivy, which is visually near-identical to
`eczema`, so the model was being punished for a boundary that barely exists in the
pixels. Those folders are now excluded. **Say so in your write-up** — "5 classes drawn
from 8 DermNet categories, with 2 visually ambiguous categories excluded" is a
defensible modelling decision, and hiding it would not be.

> ⚠️ **Not a medical device.** Course/portfolio project. Never present the output as
> medical advice.

## Step 0 — Environment check

In [ ]:
import sys, subprocess, platform
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub", "gradio"],
               check=False)

import tensorflow as tf
import numpy as np, sklearn

print("Python      :", platform.python_version())
print("TensorFlow  :", tf.__version__)
print("Keras       :", tf.keras.__version__)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    details = tf.config.experimental.get_device_details(gpus[0])
    print("\nGPU         :", details.get("device_name", gpus[0].name))
    cc = details.get("compute_capability")
    print("Compute cap :", cc, "-- float16 tensor cores" if cc and cc >= (7, 0) else "")
else:
    print("\nNO GPU. Runtime -> Change runtime type -> T4 GPU.")

## Step 1 — Configuration

### Why some DermNet folders are excluded

`rash` and `birthmark` have no single DermNet category, so they are assembled from
several. That is fine when the sources look alike and harmful when they don't:

| Class | Included | Excluded, and why |
|---|---|---|
| `rash` | Urticaria/Hives, Exanthems & Drug Eruptions | **Contact Dermatitis / Poison Ivy** — clinically and visually a form of eczema; keeping it forced the model to split an almost non-existent boundary |
| `birthmark` | Nevi & Moles, Vascular Tumors | **Pigmentation Disorders** — vitiligo is *de*pigmented (white patches), the opposite of a pigmented birthmark |

Both are in `DROPPED_KEYWORDS` below. Move an entry back into `CLASS_KEYWORDS` to
include it and watch the confusion matrix — you should see exactly the collapse the
exclusion prevents.

In [ ]:
from pathlib import Path

CONFIG = {
    "backbone":        "efficientnetv2b0",   # see BACKBONES; "mobilenetv2" for the old one
    "img_size":        224,
    "batch_size":      64,        # larger batch keeps the T4 busier
    "mixed_precision": True,      # float16 tensor cores: ~1.5-2x faster on T4
    "val_split":       0.15,
    "seed":            1337,

    "epochs_head":     8,         # frozen backbone plateaus fast
    "epochs_finetune": 25,
    "lr_head":         1e-3,
    "lr_finetune":     1e-4,      # 10x the old value, decayed by cosine -- the old
                                  # 1e-5 was too small to move the backbone at all
    "unfreeze_frac":   0.35,      # freeze the bottom 35% of the backbone
    "dropout":         0.35,
    "shuffle_buffer":  1500,      # in IMAGES (dataset is unbatched here) ~0.9 GB
    "max_per_class":   None,      # e.g. 400 for a fast smoke run

    "work_dir":   Path("/content/skin_data"),
    "cache_dir":  Path("/content/tf_cache"),
    "export_dir": Path("/content/exported_model"),
}

# name -> (constructor, external preprocessing fn or None, native resolution)
BACKBONES = {
    "efficientnetv2b0": (
        lambda shape: tf.keras.applications.EfficientNetV2B0(
            input_shape=shape, include_top=False, weights="imagenet",
            include_preprocessing=True),   # normalization is inside the backbone
        None, 224),
    "efficientnetv2b2": (
        lambda shape: tf.keras.applications.EfficientNetV2B2(
            input_shape=shape, include_top=False, weights="imagenet",
            include_preprocessing=True),
        None, 260),
    "mobilenetv2": (
        lambda shape: tf.keras.applications.MobileNetV2(
            input_shape=shape, include_top=False, weights="imagenet"),
        tf.keras.applications.mobilenet_v2.preprocess_input, 224),
}

CLASS_KEYWORDS = {
    "acne":      ["acne", "rosacea"],
    "wart":      ["wart", "molluscum"],
    "eczema":    ["eczema", "atopic dermatitis"],
    "birthmark": ["nevi", "moles", "vascular tumor"],
    "rash":      ["urticaria", "hives", "exanthem", "drug eruption"],
}

# Move any of these back into CLASS_KEYWORDS to include that category again.
DROPPED_KEYWORDS = {
    "rash":      ["contact dermatitis", "poison ivy"],
    "birthmark": ["pigmentation"],
}

CLASS_NAMES = list(CLASS_KEYWORDS.keys())
NUM_CLASSES = len(CLASS_NAMES)

for d in ("work_dir", "cache_dir", "export_dir"):
    CONFIG[d].mkdir(parents=True, exist_ok=True)

print(f"{NUM_CLASSES} classes: {CLASS_NAMES}")
print(f"backbone: {CONFIG['backbone']}  |  batch {CONFIG['batch_size']}  "
      f"|  mixed precision {CONFIG['mixed_precision']}")

## Step 2 — Download the dataset

**No Kaggle login.** DermNet is public, and `kagglehub` serves public datasets
anonymously. The cell only falls back to prompting for a token if the anonymous
request is refused — which normally never happens.

~1.9 GB, a few minutes.

In [ ]:
import os, kagglehub

def download_dermnet():
    try:
        return Path(kagglehub.dataset_download("shubhamgoel27/dermnet"))
    except Exception as e:
        print(f"Anonymous download failed ({type(e).__name__}: {e})")
        print("Falling back to an API token -- kaggle.com/settings/account -> Create New Token")
        kagglehub.login()
        return Path(kagglehub.dataset_download("shubhamgoel27/dermnet"))

RAW_PATH = download_dermnet()
print("Downloaded to:", RAW_PATH)

def find_split_root(root: Path) -> Path:
    for c in [root, *[p for p in root.iterdir() if p.is_dir()]]:
        if (c / "train").is_dir() and (c / "test").is_dir():
            return c
    raise FileNotFoundError(f"Could not locate train/ and test/ under {root}")

SPLIT_ROOT = find_split_root(RAW_PATH)
print("Split root   :", SPLIT_ROOT)
print(f"\n{len(list((SPLIT_ROOT / 'train').iterdir()))} DermNet categories found.")

### Map DermNet categories onto the 5 classes

Symlinks, so nothing is copied. The cell prints which categories fed each class and
which were deliberately skipped, so the exclusions above stay visible rather than
buried in config.

In [ ]:
import os, shutil, random
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def resolve_class(folder_name: str):
    low = folder_name.lower()
    for target, keywords in CLASS_KEYWORDS.items():
        if any(k in low for k in keywords):
            return target
    return None

def is_deliberately_dropped(folder_name: str):
    low = folder_name.lower()
    for target, keywords in DROPPED_KEYWORDS.items():
        if any(k in low for k in keywords):
            return target
    return None

def build_split(src_split: Path, dst_split: Path, cap=None):
    if dst_split.exists():
        shutil.rmtree(dst_split)
    for c in CLASS_NAMES:
        (dst_split / c).mkdir(parents=True, exist_ok=True)

    mapping, dropped, counts = {}, {}, {c: 0 for c in CLASS_NAMES}
    for folder in sorted(p for p in src_split.iterdir() if p.is_dir()):
        target = resolve_class(folder.name)
        if target is None:
            d = is_deliberately_dropped(folder.name)
            if d:
                dropped.setdefault(d, []).append(folder.name)
            continue
        mapping.setdefault(target, []).append(folder.name)

        files = sorted(f for f in folder.iterdir() if f.suffix.lower() in IMG_EXT)
        random.Random(CONFIG["seed"]).shuffle(files)
        for f in files:
            if cap is not None and counts[target] >= cap:
                break
            link = dst_split / target / f"{folder.name[:24]}__{f.name}"
            if not link.exists():
                os.symlink(f.resolve(), link)
                counts[target] += 1
    return mapping, dropped, counts

TRAIN_DIR = CONFIG["work_dir"] / "train"
TEST_DIR  = CONFIG["work_dir"] / "test"

mapping, dropped, train_counts = build_split(
    SPLIT_ROOT / "train", TRAIN_DIR, CONFIG["max_per_class"])
_, _, test_counts = build_split(SPLIT_ROOT / "test", TEST_DIR, None)

print("Categories used")
print("-" * 66)
for c in CLASS_NAMES:
    print(f"{c:>10} <- {', '.join(mapping.get(c, ['(none matched!)']))}")

if dropped:
    print("\nDeliberately excluded (see Step 1)")
    print("-" * 66)
    for c, folders in dropped.items():
        print(f"{c:>10} xx {', '.join(folders)}")

print("\nImage counts")
print("-" * 66)
print(f"{'class':>10} {'train':>8} {'test':>8}")
for c in CLASS_NAMES:
    print(f"{c:>10} {train_counts[c]:>8} {test_counts[c]:>8}")
print(f"{'TOTAL':>10} {sum(train_counts.values()):>8} {sum(test_counts.values()):>8}")

imbalance = max(train_counts.values()) / max(1, min(train_counts.values()))
print(f"\nImbalance ratio (largest/smallest): {imbalance:.1f}x -- handled by class weights")

## Step 3 — Input pipeline

The first version had two bugs here that between them exhausted Colab's 12.7 GB of
system RAM and got the runtime killed mid-training:

- **`.cache()` with no argument caches into RAM.** A decoded 224×224×3 float32 image
  is 0.6 MB, so ~10k images is ~6 GB.
- **`.shuffle(N)` on an already-batched dataset buffers N *batches*, not N images** —
  `shuffle(1000)` meant up to 32,000 images.

Fixed by caching to **disk** (`/content/tf_cache`, a few GB of the ~100 GB local SSD)
and shuffling **before** batching, so the buffer is counted in images. The disk cache
also fixes GPU starvation: JPEGs are decoded once during warm-up, then every later
epoch reads decoded tensors straight off SSD and the GPU stops waiting on the CPU.

In [ ]:
import shutil
import tensorflow as tf
AUTOTUNE = tf.data.AUTOTUNE

BACKBONE_FN, PREPROCESS, NATIVE_SIZE = BACKBONES[CONFIG["backbone"]]
if CONFIG["img_size"] != NATIVE_SIZE:
    print(f"note: {CONFIG['backbone']} is native at {NATIVE_SIZE}px, "
          f"using {CONFIG['img_size']}px")
IMG_SIZE = (CONFIG["img_size"], CONFIG["img_size"])

# batch_size=None -> individual images, so shuffle counts IMAGES not batches
common = dict(labels="inferred", label_mode="int", class_names=CLASS_NAMES,
              image_size=IMG_SIZE, batch_size=None, seed=CONFIG["seed"])

train_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=CONFIG["val_split"], subset="training",
    shuffle=True, **common)
val_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=CONFIG["val_split"], subset="validation",
    shuffle=False, **common)
test_raw = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, shuffle=False, **common)

print("\nLabel order:", train_raw.class_names)
assert train_raw.class_names == CLASS_NAMES

# Stale cache files from an earlier run with different data would be silently wrong.
shutil.rmtree(CONFIG["cache_dir"], ignore_errors=True)
CONFIG["cache_dir"].mkdir(parents=True, exist_ok=True)

def pipeline(ds, name, training):
    ds = ds.cache(str(CONFIG["cache_dir"] / name))     # DISK, not RAM
    if training:
        ds = ds.shuffle(CONFIG["shuffle_buffer"], seed=CONFIG["seed"],
                        reshuffle_each_iteration=True)
    return ds.batch(CONFIG["batch_size"]).prefetch(AUTOTUNE)

train_ds = pipeline(train_raw, "train", True)
val_ds   = pipeline(val_raw,   "val",   False)
test_ds  = pipeline(test_raw,  "test",  False)

# "Enhance quality" -- runs on the GPU as model layers, training only.
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.15),
    tf.keras.layers.RandomBrightness(0.15, value_range=(0, 255)),
], name="augmentation")

In [ ]:
import time
# Warm the disk cache once. Every epoch after this reads decoded tensors off SSD
# instead of re-decoding JPEGs, which is what keeps the GPU fed.
for name, ds in (("train", train_ds), ("val", val_ds), ("test", test_ds)):
    t0, n = time.time(), 0
    for xb, _ in ds:
        n += int(xb.shape[0])
    print(f"cached {name:>5}: {n:>6} images in {time.time() - t0:6.1f}s")

cache_mb = sum(f.stat().st_size for f in CONFIG["cache_dir"].glob("*")) / 1e6
print(f"\ncache on disk: {cache_mb:,.0f} MB (RAM usage stays flat)")

In [ ]:
import matplotlib.pyplot as plt

images, labels = next(iter(train_ds.take(1)))
fig, axes = plt.subplots(2, 6, figsize=(16, 5.5))
for i in range(6):
    axes[0, i].imshow(images[i].numpy().astype("uint8"))
    axes[0, i].set_title(CLASS_NAMES[labels[i]], fontsize=10)
    aug = data_augmentation(tf.expand_dims(images[i], 0), training=True)[0]
    axes[1, i].imshow(tf.cast(tf.clip_by_value(aug, 0, 255), tf.uint8).numpy())
    axes[1, i].set_title("augmented", fontsize=9, color="gray")
for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Top: original   |   Bottom: after augmentation", y=1.0)
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Labels come from the file paths -- no need to decode a single image for this.
train_labels = np.array([CLASS_NAMES.index(Path(p).parent.name)
                         for p in train_raw.file_paths])
present = np.unique(train_labels)
weights = compute_class_weight("balanced", classes=present, y=train_labels)
CLASS_WEIGHT = {int(c): float(w) for c, w in zip(present, weights)}

print(f"{'class':>10} {'n':>7} {'weight':>8}")
for i, c in enumerate(CLASS_NAMES):
    print(f"{c:>10} {int((train_labels == i).sum()):>7} {CLASS_WEIGHT.get(i, 0.0):>8.3f}")

## Step 4 — Model

EfficientNetV2B0 with a fresh head. Three things differ from the first version:

- **Mixed precision.** The T4 has float16 tensor cores; the compute runs in float16
  while weights stay float32. The softmax is forced back to float32 — in float16 it
  can overflow and produce `nan` loss.
- **Preprocessing is inside the model.** EfficientNetV2 carries its own normalization
  (`include_preprocessing=True`), so exported models take raw 0–255 images either way.
- **`base_model(x, training=False)`** keeps BatchNorm in inference mode. Letting BN
  statistics update while fine-tuning on a small dataset is the classic cause of
  "validation accuracy collapses the moment I unfreeze".

In [ ]:
from tensorflow.keras import layers, Model

if CONFIG["mixed_precision"] and tf.config.list_physical_devices("GPU"):
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("compute dtype:", tf.keras.mixed_precision.global_policy().compute_dtype)

def build_model():
    base_model = BACKBONE_FN(IMG_SIZE + (3,))
    base_model.trainable = False

    inputs = layers.Input(shape=IMG_SIZE + (3,), name="image")
    x = data_augmentation(inputs)
    if PREPROCESS is not None:
        x = PREPROCESS(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D(name="pool")(x)
    x = layers.Dropout(CONFIG["dropout"], name="dropout")(x)
    x = layers.Dense(NUM_CLASSES, name="logits")(x)
    # float32 softmax: float16 overflows here and gives nan loss
    outputs = layers.Activation("softmax", dtype="float32", name="predictions")(x)
    return Model(inputs, outputs, name="skin_disease_classifier"), base_model

model, base_model = build_model()
model.compile(optimizer=tf.keras.optimizers.Adam(CONFIG["lr_head"]),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=["accuracy"])

print(f"backbone layers: {len(base_model.layers)}")
print(f"trainable params: {sum(int(tf.size(w)) for w in model.trainable_weights):,} "
      f"/ {sum(int(tf.size(w)) for w in model.weights):,}")

### Phase 1 — train the head

Only the new `Dense` learns. This plateaus quickly; that plateau is the ceiling of
frozen ImageNet features on dermatology images, and it is roughly where the first
version of this notebook stopped improving.

In [ ]:
CKPT = str(CONFIG["export_dir"] / "best.keras")

history_head = model.fit(
    train_ds, validation_data=val_ds,
    epochs=CONFIG["epochs_head"],
    class_weight=CLASS_WEIGHT,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(CKPT, monitor="val_accuracy", mode="max",
                                           save_best_only=True, verbose=0),
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=4,
                                         restore_best_weights=True, verbose=1),
    ],
)
print(f"\nphase 1 best val accuracy: {max(history_head.history['val_accuracy']):.2%}")

### Phase 2 — fine-tune

This is where the accuracy actually comes from. Two changes from the first version:

- **Unfreeze the top 65% of the backbone** rather than a fixed layer index.
- **LR 1e-4 with cosine decay to ~1e-6**, instead of a flat 1e-5. The old value was
  small enough that the backbone barely moved — which is why accuracy stalled near
  where phase 1 left it.

`ReduceLROnPlateau` is gone: it conflicts with a schedule (Keras raises if you try to
set the LR on an optimizer that owns a `LearningRateSchedule`).

In [ ]:
cut = int(len(base_model.layers) * CONFIG["unfreeze_frac"])
base_model.trainable = True
for layer in base_model.layers[:cut]:
    layer.trainable = False
for layer in base_model.layers:                      # BN frozen everywhere
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

steps_per_epoch = len(train_ds)
schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=CONFIG["lr_finetune"],
    decay_steps=steps_per_epoch * CONFIG["epochs_finetune"],
    alpha=0.01)                                      # floor at 1% of the initial LR

model.compile(optimizer=tf.keras.optimizers.Adam(schedule),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=["accuracy"])

print(f"unfrozen from layer {cut}/{len(base_model.layers)}  |  "
      f"trainable params: {sum(int(tf.size(w)) for w in model.trainable_weights):,}")

history_ft = model.fit(
    train_ds, validation_data=val_ds,
    epochs=CONFIG["epochs_head"] + CONFIG["epochs_finetune"],
    initial_epoch=len(history_head.history["loss"]),
    class_weight=CLASS_WEIGHT,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(CKPT, monitor="val_accuracy", mode="max",
                                           save_best_only=True, verbose=0),
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=8,
                                         restore_best_weights=True, verbose=1),
    ],
)

In [ ]:
def merge(k): return history_head.history.get(k, []) + history_ft.history.get(k, [])
acc, val_acc   = merge("accuracy"), merge("val_accuracy")
loss, val_loss = merge("loss"),     merge("val_loss")
split = len(history_head.history["loss"])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, tr, va, name in [(a1, acc, val_acc, "Accuracy"), (a2, loss, val_loss, "Loss")]:
    ax.plot(tr, label="train", linewidth=2)
    ax.plot(va, label="validation", linewidth=2)
    ax.axvline(split - 0.5, color="crimson", ls="--", alpha=0.7, label="fine-tuning starts")
    ax.set_xlabel("epoch"); ax.set_title(name); ax.legend(); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

print(f"Best validation accuracy: {max(val_acc):.2%} (epoch {int(np.argmax(val_acc)) + 1})")
print(f"Gain from fine-tuning   : {max(val_acc) - max(val_acc[:split]):+.2%}")

## Step 5 — Evaluate

Test-time augmentation averages the prediction over the image plus its two flips.
Costs three forward passes, usually worth 1–2 points, and needs no retraining.

**Read the per-class F1, not just overall accuracy.** With imbalanced data a model can
look fine overall and be useless on the smallest class.

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, f1_score)

def predict_tta(m, ds):
    views = [lambda x: x, tf.image.flip_left_right, tf.image.flip_up_down]
    return np.mean([m.predict(ds.map(lambda x, y: (fn(x), y)), verbose=0)
                    for fn in views], axis=0)

y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)

y_prob_plain = model.predict(test_ds, verbose=0)
y_prob       = predict_tta(model, test_ds)

acc_plain = (np.argmax(y_prob_plain, 1) == y_true).mean()
y_pred    = np.argmax(y_prob, axis=1)
test_acc  = (y_pred == y_true).mean()

print(f"Test accuracy (plain): {acc_plain:.2%}")
print(f"Test accuracy (TTA)  : {test_acc:.2%}   ({test_acc - acc_plain:+.2%})\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))
print(f"Macro F1   : {f1_score(y_true, y_pred, average='macro'):.3f}")
print(f"Weighted F1: {f1_score(y_true, y_pred, average='weighted'):.3f}")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 6))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=CLASS_NAMES
    ).plot(ax=a1, cmap="Blues", colorbar=False, values_format="d")
a1.set_title("Confusion matrix (counts)")
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, normalize="true"),
                       display_labels=CLASS_NAMES
    ).plot(ax=a2, cmap="Blues", colorbar=False, values_format=".2f")
a2.set_title("Normalized by true class (recall on the diagonal)")
for ax in (a1, a2):
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

# The worst off-diagonal cell tells you which pair to attack next.
cm = confusion_matrix(y_true, y_pred, normalize="true").copy()
np.fill_diagonal(cm, 0)
i, j = np.unravel_index(np.argmax(cm), cm.shape)
print(f"Biggest confusion: {CLASS_NAMES[i]} predicted as {CLASS_NAMES[j]} "
      f"({cm[i, j]:.0%} of all true {CLASS_NAMES[i]})")

In [ ]:
wrong = np.where(y_pred != y_true)[0]
print(f"{len(wrong)} misclassified out of {len(y_true)} ({len(wrong)/len(y_true):.1%})")
show = wrong[np.argsort(-y_prob[wrong].max(axis=1))][:8]

# Fetch only the few images we display -- concatenating the whole test set would
# put ~1 GB of float32 pixels in RAM for eight thumbnails.
need, picked, idx = set(int(i) for i in show), {}, 0
for xb, _ in test_ds:
    for img in xb.numpy():
        if idx in need:
            picked[idx] = img
        idx += 1
    if len(picked) == len(need):
        break

if len(show):
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax, i in zip(axes.flat, show):
        ax.imshow(picked[int(i)].astype("uint8"))
        ax.set_title(f"true: {CLASS_NAMES[y_true[i]]}\n"
                     f"pred: {CLASS_NAMES[y_pred[i]]} ({y_prob[i].max():.0%})",
                     fontsize=10, color="crimson")
        ax.axis("off")
    for ax in axes.flat[len(show):]:
        ax.axis("off")
    plt.suptitle("Most confident mistakes", y=1.0)
    plt.tight_layout(); plt.show()

## Step 6 — Export

The `.keras` file keeps the augmentation layers so training can resume. The TFLite
build strips them: TFLite has no native kernels for the random-transform ops
(`ImageProjectiveTransformV3`, `StatelessRandomUniformV2`, `Tan`) and conversion of
the full model fails outright. The cell asserts both graphs agree before converting.

Both take raw 0–255 RGB at the configured resolution.

In [ ]:
import json
EXPORT = CONFIG["export_dir"]

keras_path = EXPORT / "skin_classifier.keras"
model.save(keras_path)
print(f"Keras model : {keras_path.name}  ({keras_path.stat().st_size / 1e6:.1f} MB)")

inf_in = tf.keras.Input(shape=IMG_SIZE + (3,), name="image")
z = inf_in
if PREPROCESS is not None:
    z = PREPROCESS(z)
z = base_model(z, training=False)
for name in ("pool", "logits", "predictions"):
    z = model.get_layer(name)(z)
inference_model = tf.keras.Model(inf_in, z, name="skin_classifier_inference")

probe = np.random.randint(0, 256, (4,) + IMG_SIZE + (3,)).astype("float32")
np.testing.assert_allclose(model.predict(probe, verbose=0),
                           inference_model.predict(probe, verbose=0), atol=1e-4)
print("Inference graph matches the trained model")

labels = {
    "class_names": CLASS_NAMES,
    "backbone": CONFIG["backbone"],
    "input_shape": [CONFIG["img_size"], CONFIG["img_size"], 3],
    "input_range": "0-255 RGB (preprocessing is inside the model)",
    "excluded_categories": DROPPED_KEYWORDS,
    "metrics": {
        "test_accuracy_tta": float(test_acc),
        "test_accuracy_plain": float(acc_plain),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "per_class_f1": {c: float(s) for c, s in
                         zip(CLASS_NAMES, f1_score(y_true, y_pred, average=None))},
    },
}
(EXPORT / "labels.json").write_text(json.dumps(labels, indent=2))
print("Labels      : labels.json")

tflite_path = EXPORT / "skin_classifier.tflite"
try:
    sm_dir = "/content/_saved_model"
    inference_model.export(sm_dir)
    conv = tf.lite.TFLiteConverter.from_saved_model(sm_dir)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_path.write_bytes(conv.convert())
    print(f"TFLite      : {tflite_path.name}  ({tflite_path.stat().st_size / 1e6:.1f} MB)")
except Exception as e:
    print(f"TFLite conversion skipped: {type(e).__name__}: {e}")

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("/content/skin_classifier_export", "zip", EXPORT)
files.download(archive)

## Step 7 — Demo

In [ ]:
import gradio as gr
from PIL import Image

def predict(img):
    if img is None:
        return {}
    im = Image.fromarray(img).convert("RGB").resize(IMG_SIZE)
    arr = np.expand_dims(np.array(im, dtype="float32"), 0)
    probs = np.mean([inference_model.predict(v, verbose=0)[0] for v in
                     (arr, arr[:, :, ::-1], arr[:, ::-1])], axis=0)   # TTA
    return {c: float(p) for c, p in zip(CLASS_NAMES, probs)}

gr.Interface(
    fn=predict,
    inputs=gr.Image(type="numpy", label="Skin image"),
    outputs=gr.Label(num_top_classes=NUM_CLASSES, label="Prediction"),
    title="Skin Disease Classifier",
    description=(f"{CONFIG['backbone']} fine-tuned on DermNet. "
                 f"Test accuracy {test_acc:.1%}. Educational project only -- "
                 "not a diagnostic tool."),
    flagging_mode="never",
).launch(share=True, debug=False)

## If it still falls short of 70%

Work the confusion matrix, not the hyperparameters. The cell above prints the worst
confused pair — that one number tells you which of these to reach for:

1. **Two classes bleed into each other.** They may not be separable from a photo.
   Check `DROPPED_KEYWORDS` — you may need to exclude another category, or merge the
   pair into one class and report 4 classes honestly.
2. **One class has poor recall but no single confusor.** Not enough data. Raise its
   weight, or add a source category to its `CLASS_KEYWORDS` entry.
3. **Train accuracy ≫ validation.** Overfitting: raise `dropout` toward 0.5, or
   strengthen augmentation.
4. **Both accuracies are low and still climbing at the end.** Underfitting: raise
   `epochs_finetune`, or lower `unfreeze_frac` to 0.2 to free more of the backbone.

Bigger levers, in order of payoff:

- `"backbone": "efficientnetv2b2"` with `"img_size": 260` — lesion texture is
  fine-grained and 224px discards detail. Costs roughly 2x the training time.
- **Cross-validation** — with a test set this size, a single split moves ±2–3% run to
  run. Report a mean over folds if you are quoting a number.

**What not to do:** don't tune anything against the test set. Use validation accuracy
for every decision and touch the test set once, at the end. Selecting a backbone by
test accuracy silently inflates the number you report.